In [ ]:
!pip install -q transformers accelerate bitsandbytes bert-score sacrebleu nltk sentencepiece

In [ ]:
#!/usr/bin/env python3
# pip install -q transformers accelerate bitsandbytes bert-score sacrebleu nltk sentencepiece
import os
import gc
import traceback
import warnings
import torch
import pandas as pd
import nltk
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from bert_score import score as bert_score

nltk.download("punkt", quiet=True)

warnings.filterwarnings("ignore", message=".*will be cast from.*")
import logging
logging.getLogger("bitsandbytes").setLevel(logging.ERROR)

CSV_PATH    = "/kaggle/input/datasets/syedmdnafissameen/chqsusss/CHQSUMM_sampled_200.csv"
MODEL_ID    = "md-nishat-008/TigerLLM-9B-it"
OUTPUT_DIR  = "/kaggle/working/bangla_chq_benchmark"
NUM_SHOTS   = 3
MAX_NEW_TOKENS = 128
BATCH_SIZE  = 1  # generation batch (keep 1 for TigerLLM stability on T4)

os.makedirs(OUTPUT_DIR, exist_ok=True)

PAPER_SCORES = {
    "ROUGE-1": 50.05,
    "ROUGE-2": 29.11,
    "ROUGE-L": 48.35,
    "BERTScore": 89.91,
}

SYSTEM_PROMPT = """You are a Bengali medical question summarization model.
Given a Bengali health-related question asked by a patient, generate a concise summary.
Rules:
- Retain all medically relevant information required to answer the question accurately.
- Be as concise as possible without discarding essential information.
- Preserve symptom details, duration, medications mentioned, and the core question.
- Output ONLY the Bengali summary, nothing else.
- Do not include explanations, labels, or any extra text."""

# ---------------- Load dataset & pick few-shot exemplars ----------------
df_full = pd.read_csv(CSV_PATH)
assert {"question", "summary"}.issubset(df_full.columns)

FEWSHOT_IDS = df_full.index[:NUM_SHOTS].tolist()
fewshot_examples = df_full.loc[FEWSHOT_IDS, ["question", "summary"]].to_dict("records")
df = df_full.drop(index=FEWSHOT_IDS).reset_index(drop=True)

print(f"Using {NUM_SHOTS} few-shot exemplars, {len(df)} rows to evaluate.")

# ---------------- Load model in 8-bit sharded across 2x T4 ----------------
# Single-T4 (14.5GB usable) OOMs for this checkpoint in 8-bit, so shard across both.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
gc.collect()
torch.cuda.empty_cache()

bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True,  # safety valve if it doesn't fit even sharded
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

max_memory = {0: "13GiB", 1: "13GiB", "cpu": "48GiB"}

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=max_memory,
    trust_remote_code=True,
)
model.eval()

HAS_CHAT_TEMPLATE = getattr(tokenizer, "chat_template", None) is not None

# ---------------- Prompt construction ----------------
def build_messages(question: str):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in fewshot_examples:
        messages.append({"role": "user", "content": f"Summarize this Bengali health question:\n{ex['question']}"})
        messages.append({"role": "assistant", "content": ex["summary"]})
    messages.append({"role": "user", "content": f"Summarize this Bengali health question:\n{question}"})
    return messages


def build_plain_prompt(question: str) -> str:
    """Fallback manual prompt if the tokenizer has no usable chat template."""
    parts = [SYSTEM_PROMPT, ""]
    for ex in fewshot_examples:
        parts.append(f"### Question:\n{ex['question']}\n### Summary:\n{ex['summary']}")
    parts.append(f"### Question:\n{question}\n### Summary:\n")
    return "\n\n".join(parts)


def generate(question: str) -> str:
    if HAS_CHAT_TEMPLATE:
        messages = build_messages(question)
        templated = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        )
        # Some chat templates / tokenizer configs return a BatchEncoding (dict-like)
        # instead of a raw tensor — normalize to a tensor either way.
        if isinstance(templated, torch.Tensor):
            input_ids = templated.to(model.device)
        else:
            input_ids = templated["input_ids"].to(model.device)
    else:
        prompt = build_plain_prompt(question)
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)

    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    gen_tokens = output[0][input_ids.shape[-1]:]
    text = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    if "<think>" in text:
        text = text.split("</think>")[-1].strip()
    return text


# ---------------- Checkpointing ----------------
def checkpoint_path():
    return os.path.join(OUTPUT_DIR, f"checkpoint_{MODEL_ID.replace('/', '__')}.csv")

def load_checkpoint():
    path = checkpoint_path()
    return pd.read_csv(path) if os.path.exists(path) else None

def save_checkpoint(rows):
    pd.DataFrame(rows).to_csv(checkpoint_path(), index=False, encoding="utf-8-sig")


# ---------------- Run inference ----------------
def sanity_check():
    """Run one generation eagerly so a bad chat template / device issue surfaces
    immediately with a full traceback instead of silently failing 200 times."""
    print("Running sanity check on 1 example...")
    test_q = df.iloc[0]["question"]
    out = generate(test_q)
    print(f"Sanity check output: {out[:200]!r}")


def run_inference():
    sanity_check()
    existing = load_checkpoint()
    rows_done = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows_done)

    if start_idx >= len(df):
        print("Already complete — loaded from checkpoint.")
        return rows_done

    for idx in range(start_idx, len(df)):
        row = df.iloc[idx]
        question = str(row["question"])
        reference = str(row["summary"])

        try:
            prediction = generate(question)
            err = None
        except Exception:
            prediction = ""
            err = "gen_error:" + traceback.format_exc(limit=3).replace("\n", " | ")

        rows_done.append({
            "id": row.get("id", idx),
            "question": question,
            "reference": reference,
            "prediction": prediction,
            "error": err,
        })

        print(f"[{idx+1}/{len(df)}] err={err} | q={question[:50]}...")

        if (idx + 1) % 10 == 0 or idx == len(df) - 1:
            save_checkpoint(rows_done)
            torch.cuda.empty_cache()
            gc.collect()

    save_checkpoint(rows_done)
    return rows_done


# ---------------- Metrics (same as reference script) ----------------
def _ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))

def _rouge_n(pred_tokens, ref_tokens, n):
    pred_ng = _ngrams(pred_tokens, n)
    ref_ng  = _ngrams(ref_tokens,  n)
    overlap = sum((pred_ng & ref_ng).values())
    p = overlap / max(sum(pred_ng.values()), 1)
    r = overlap / max(sum(ref_ng.values()),  1)
    f = (2 * p * r) / max(p + r, 1e-9)
    return f

def _lcs_len(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            dp[i][j] = dp[i-1][j-1] + 1 if a[i-1] == b[j-1] else max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

def _rouge_l(pred_tokens, ref_tokens):
    lcs = _lcs_len(pred_tokens, ref_tokens)
    p = lcs / max(len(pred_tokens), 1)
    r = lcs / max(len(ref_tokens),  1)
    f = (2 * p * r) / max(p + r, 1e-9)
    return f


def compute_metrics(rows):
    predictions, references = [], []
    for r in rows:
        pred = str(r.get("prediction", "")).strip()
        ref  = str(r.get("reference",  "")).strip()
        err  = r.get("error")
        if pred and ref and (err is None or str(err).strip().lower() in ("", "nan", "none")):
            predictions.append(pred)
            references.append(ref)

    if not predictions:
        print("No valid predictions found.")
        return {}

    print(f"Evaluating {len(predictions)} valid predictions...")

    r1, r2, rl = [], [], []
    for pred, ref in zip(predictions, references):
        pt = pred.split()
        rt = ref.split()
        r1.append(_rouge_n(pt, rt, 1))
        r2.append(_rouge_n(pt, rt, 2))
        rl.append(_rouge_l(pt, rt))

    rouge1 = (sum(r1) / len(r1)) * 100
    rouge2 = (sum(r2) / len(r2)) * 100
    rougel = (sum(rl) / len(rl)) * 100

    predictions_trunc = [" ".join(p.split()[:200]) for p in predictions]
    references_trunc  = [" ".join(r.split()[:200]) for r in references]

    print("Computing BERTScore with BanglaBERT (this may take a while)...")
    P, R, F1 = bert_score(
        predictions_trunc,
        references_trunc,
        model_type="xlm-roberta-base",
        lang="bn",
        verbose=False,
    )
    bertscore = F1.mean().item() * 100

    return {
        "ROUGE-1":   rouge1,
        "ROUGE-2":   rouge2,
        "ROUGE-L":   rougel,
        "BERTScore": bertscore,
        "total":     len(predictions),
    }


def main():
    rows = run_inference()

    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, "predictions.csv"), index=False, encoding="utf-8-sig")

    metrics = compute_metrics(rows)
    if not metrics:
        return

    results_df = pd.DataFrame([{
        "model": MODEL_ID,
        "shots": NUM_SHOTS,
        "ROUGE-1":   round(metrics["ROUGE-1"], 2),
        "ROUGE-2":   round(metrics["ROUGE-2"], 2),
        "ROUGE-L":   round(metrics["ROUGE-L"], 2),
        "BERTScore": round(metrics["BERTScore"], 2),
        "total_sentences": metrics["total"],
    }])
    results_df.to_csv(
        os.path.join(OUTPUT_DIR, "benchmark_results.csv"), index=False, encoding="utf-8-sig")

    print("\n── Evaluation Results (paper Table 2 format) ──")
    print(f"  {'Metric':<12} {'Yours':>8}  {'Paper (BanglaT5)':>18}  {'Diff':>8}")
    print(f"  {'-'*52}")
    for metric in ["ROUGE-1", "ROUGE-2", "ROUGE-L", "BERTScore"]:
        your  = metrics[metric]
        paper = PAPER_SCORES[metric]
        diff  = your - paper
        sign  = "+" if diff >= 0 else ""
        print(f"  {metric:<12} {your:>8.2f}  {paper:>18.2f}  {sign}{diff:>7.2f}")

    print(f"\n  Total evaluated: {metrics['total']} summaries")


if __name__ == "__main__":
    main()